# ZemaVoice Stage 4 — Amharic speech baseline
This notebook **tests an existing pretrained Amharic model**; it does not train or own a new model. No ZIP upload required. The model `facebook/mms-tts-amh` is **CC BY-NC 4.0: noncommercial evaluation only**. Do not deploy it in a paid/commercial ZemaVoice product without a suitable license.
Run cells from top to bottom. CPU is sufficient; T4 optional.

In [ ]:
!pip -q install 'transformers>=4.44,<5' 'accelerate>=0.26' 'soundfile>=0.12'

In [ ]:
import os, subprocess, torch, soundfile as sf
from IPython.display import Audio, display
from transformers import VitsModel, VitsTokenizer, set_seed
if not os.path.isfile('/content/uroman/bin/uroman.pl'):
    subprocess.run(['git','clone','--depth','1','https://github.com/isi-nlp/uroman.git','/content/uroman'],check=True)
model_id='facebook/mms-tts-amh'
tokenizer=VitsTokenizer.from_pretrained(model_id)
model=VitsModel.from_pretrained(model_id)
device='cuda' if torch.cuda.is_available() else 'cpu'
model=model.to(device).eval()
print('Model ready on',device,'; uroman required:',tokenizer.is_uroman)

In [ ]:
# Edit this Amharic sentence for your first test.
amharic_text='ሰላም እንዴት ነህ ወደ ዜማ ቮይስ እንኳን ደህና መጣህ'
p=subprocess.run(['perl','/content/uroman/bin/uroman.pl'],input=amharic_text,text=True,capture_output=True,check=True)
romanized=p.stdout.strip()
print('Romanized input:',romanized)
assert romanized, 'Romanization failed'
inputs=tokenizer(romanized,return_tensors='pt').to(device)
set_seed(42)
with torch.no_grad():
    waveform=model(**inputs).waveform[0].detach().cpu().float().numpy()
sample_rate=model.config.sampling_rate
sf.write('/content/zemavoice_stage4_amharic_test.wav',waveform,sample_rate)
display(Audio(waveform,rate=sample_rate))
print('Saved /content/zemavoice_stage4_amharic_test.wav')

In [ ]:
from google.colab import files
files.download('/content/zemavoice_stage4_amharic_test.wav')

## Next
Listen for intelligible Amharic, note pronunciation errors, and test several phrases. This pretrained model is single-speaker and **not** a 20+ voice product. A commercial version needs appropriately licensed model weights and recordings, explicit voice consent, and additional engineering.